# 키네마틱 피처 기반 파킨슨병 분류 파이프라인
- **데이터**: UCI (PD 25, HC 15) + PaHaW (PD 37, HC 38) = 115명
- **피처**: velocity/acceleration/jerk 통계량 12개
- **전처리**: Fold 내 소스별 정규화 (누수 없음)
- **모델**: RandomForest + 5-Fold Stratified CV
- **출력**: `kinematic_model.pkl`

## 1. 라이브러리 임포트 및 설정

In [61]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix, roc_curve
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.use('Agg')
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)
print('설정 완료')

설정 완료


## 2. 피처 추출 함수

In [62]:
def extract_kinematic_features(x, y, t):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    t = np.array(t, dtype=float)

    dt = np.diff(t) / 1000.0
    dt = np.where(dt == 0, 1e-6, dt)

    dx = np.diff(x); dy = np.diff(y)
    dist         = np.sqrt(dx**2 + dy**2)
    velocity     = dist / dt
    acceleration = np.abs(np.diff(velocity)) / dt[1:]
    jerk         = np.abs(np.diff(acceleration)) / dt[2:]

    def stats(arr):
        if len(arr) == 0: return [0.0, 0.0, 0.0]
        return [float(np.mean(arr)), float(np.std(arr)), float(np.max(arr))]

    feats = {}
    for name, arr in [('velocity', velocity), ('acceleration', acceleration), ('jerk', jerk)]:
        for i, s in enumerate(['mean', 'std', 'max']):
            feats[f'{name}_{s}'] = stats(arr)[i]

    feats['total_distance'] = float(np.sum(dist))
    feats['duration']       = float((t[-1] - t[0]) / 1000.0)
    feats['velocity_cv']    = feats['velocity_std'] / (feats['velocity_mean'] + 1e-6)
    return feats

FEATURE_COLS = [
    'velocity_mean', 'velocity_std', 'velocity_max', 'velocity_cv',
    'acceleration_mean', 'acceleration_std', 'acceleration_max',
    'jerk_mean', 'jerk_std', 'jerk_max',
    'total_distance', 'duration'
]
print(f'피처 수: {len(FEATURE_COLS)}개')

피처 수: 12개


## 3. UCI 데이터 로드

In [63]:
UCI_DIR = './UCI/hw_dataset'

def load_uci(data_dir):
    records = []
    for label, cls in [(1, 'parkinson'), (0, 'control')]:
        cls_dir = os.path.join(data_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if not fname.endswith('.txt'): continue
            df = pd.read_csv(
                os.path.join(cls_dir, fname), sep=';', header=None,
                names=['X','Y','Z','Pressure','GripAngle','Timestamp','TestID']
            )
            df = df[df['TestID'] == 0].reset_index(drop=True)
            if len(df) < 10: continue
            feats = extract_kinematic_features(df['X'].values, df['Y'].values, df['Timestamp'].values)
            feats['label'] = label
            feats['source'] = 'UCI'
            records.append(feats)
    return pd.DataFrame(records)

uci_df = load_uci(UCI_DIR)
print(f'UCI: {len(uci_df)}명  PD={uci_df.label.sum()}  HC={len(uci_df)-uci_df.label.sum()}')

UCI: 40명  PD=25  HC=15


## 4. PaHaW 데이터 로드

In [64]:
PAHAW_DIR   = 'PaHaW/PaHaW_public'
CORPUS_PATH = 'PaHaW/PaHaW_files/corpus_PaHaW.xlsx'

def load_svc(path):
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    n = int(lines[0].strip())
    rows = [list(map(float, l.strip().split())) for l in lines[1:n+1]]
    return pd.DataFrame(rows, columns=['Y','X','Timestamp','ButtonStatus','Azimuth','Altitude','Pressure'])

def load_pahaw(pahaw_dir, corpus_path):
    corpus = pd.read_excel(corpus_path)
    corpus['ID'] = corpus['ID'].astype(str).str.zfill(5)
    label_map = {r['ID']: (1 if r['Disease'] == 'PD' else 0) for _, r in corpus.iterrows()}
    records = []
    for subj_id in sorted(os.listdir(pahaw_dir)):
        subj_dir = os.path.join(pahaw_dir, subj_id)
        if not os.path.isdir(subj_dir): continue
        task8 = [f for f in os.listdir(subj_dir) if f.endswith('__8_1.svc')]
        if not task8: continue
        df = load_svc(os.path.join(subj_dir, task8[0]))
        df = df[df['ButtonStatus'] == 1].reset_index(drop=True)
        if len(df) < 10: continue
        label = label_map.get(subj_id)
        if label is None: continue
        feats = extract_kinematic_features(df['X'].values, df['Y'].values, df['Timestamp'].values)
        feats['label'] = label
        feats['source'] = 'PaHaW'
        records.append(feats)
    return pd.DataFrame(records)

pahaw_df = load_pahaw(PAHAW_DIR, CORPUS_PATH)
print(f'PaHaW: {len(pahaw_df)}명  PD={pahaw_df.label.sum()}  HC={len(pahaw_df)-pahaw_df.label.sum()}')

PaHaW: 75명  PD=37  HC=38


## 5. 데이터 합산 및 이상값 제거

In [65]:
combined_df = pd.concat([uci_df, pahaw_df], ignore_index=True)

before = len(combined_df)
combined_df = combined_df[combined_df['duration'] < 1000].reset_index(drop=True)
print(f'이상값 제거: {before} → {len(combined_df)}명')
print(combined_df.groupby(['source','label']).size().unstack().rename(columns={0:'HC',1:'PD'}))

nan_count = combined_df[FEATURE_COLS].isnull().sum().sum()
inf_count = np.isinf(combined_df[FEATURE_COLS].values).sum()
print(f'NaN: {nan_count}개  Inf: {inf_count}개')

# 원본 raw 피처 (정규화 전)
X_raw   = combined_df[FEATURE_COLS].values.copy()
y       = combined_df['label'].values
sources = combined_df['source'].values
print(f'X_raw shape: {X_raw.shape}')

이상값 제거: 115 → 114명
label   HC  PD
source        
PaHaW   38  36
UCI     15  25
NaN: 0개  Inf: 0개
X_raw shape: (114, 12)


## 6. 5-Fold CV — Fold 내 소스별 정규화 (누수 없음)

> 각 fold마다:
> 1. **학습 데이터**에서만 소스별 mean/std 계산
> 2. 계산한 통계로 **학습 + 검증 데이터** 모두 정규화
> 3. 검증 데이터 정보는 정규화 계산에 절대 포함 안 됨

In [57]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Pipeline: 소스 정규화 이후 추가 StandardScaler + RF
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ))
])

fold_aucs, fold_f1s = [], []
all_probas, all_preds, all_true = [], [], []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_raw, y)):
    X_tr = X_raw[tr_idx].copy()
    X_vl = X_raw[vl_idx].copy()

    # ── Fold 내 소스별 정규화 (학습 데이터 통계만 사용) ──
    for src in ['UCI', 'PaHaW']:
        tr_src_mask = sources[tr_idx] == src
        vl_src_mask = sources[vl_idx] == src
        if tr_src_mask.sum() == 0:
            continue
        sc = StandardScaler()
        X_tr[tr_src_mask] = sc.fit_transform(X_tr[tr_src_mask])   # 학습만으로 fit
        if vl_src_mask.sum() > 0:
            X_vl[vl_src_mask] = sc.transform(X_vl[vl_src_mask])   # 검증엔 transform만

    pipeline.fit(X_tr, y[tr_idx])
    proba = pipeline.predict_proba(X_vl)[:, 1]
    pred  = pipeline.predict(X_vl)

    auc = roc_auc_score(y[vl_idx], proba)
    f1  = f1_score(y[vl_idx], pred)
    fold_aucs.append(auc)
    fold_f1s.append(f1)
    all_probas.extend(proba)
    all_preds.extend(pred)
    all_true.extend(y[vl_idx])
    print(f'Fold {fold+1}: AUC={auc:.4f}  F1={f1:.4f}')

print(f'\n평균 AUC: {np.mean(fold_aucs):.4f} (+/-{np.std(fold_aucs):.4f})')
print(f'평균 F1:  {np.mean(fold_f1s):.4f} (+/-{np.std(fold_f1s):.4f})')

Fold 1: AUC=0.8462  F1=0.8571
Fold 2: AUC=0.6667  F1=0.6667
Fold 3: AUC=0.6894  F1=0.6667
Fold 4: AUC=0.7803  F1=0.6667
Fold 5: AUC=0.8250  F1=0.8333

평균 AUC: 0.7615 (+/-0.0718)
평균 F1:  0.7381 (+/-0.0878)


## 7. 결과 시각화

In [58]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='#EEF4ED')
fpr, tpr, _ = roc_curve(all_true, all_probas)
overall_auc = roc_auc_score(all_true, all_probas)
axes[0].plot(fpr, tpr, color='#13315C', lw=2, label=f'AUC = {overall_auc:.3f}')
axes[0].plot([0,1],[0,1],'--',color='gray')
axes[0].set_title('ROC Curve (5-Fold OOF)', color='#13315C', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].set_facecolor('#F8FAF8')
cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['HC','PD'], yticklabels=['HC','PD'])
axes[1].set_title('Confusion Matrix (5-Fold OOF)', color='#13315C', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.savefig('kinematic_results.png', dpi=100, bbox_inches='tight')
plt.close()
print('저장: kinematic_results.png')
print(classification_report(all_true, all_preds, target_names=['HC','PD']))

저장: kinematic_results.png
              precision    recall  f1-score   support

          HC       0.71      0.68      0.69        53
          PD       0.73      0.75      0.74        61

    accuracy                           0.72       114
   macro avg       0.72      0.72      0.72       114
weighted avg       0.72      0.72      0.72       114



## 8. 피처 중요도

In [59]:
# 전체 데이터 소스별 정규화 후 최종 학습
X_norm_final = X_raw.copy()
final_src_scalers = {}
for src in ['UCI', 'PaHaW']:
    mask = sources == src
    sc = StandardScaler()
    X_norm_final[mask] = sc.fit_transform(X_norm_final[mask])
    final_src_scalers[src] = sc

pipeline.fit(X_norm_final, y)
importances = pd.Series(
    pipeline.named_steps['rf'].feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#EEF4ED')
importances.plot(kind='barh', ax=ax, color='#8DA9C4')
ax.set_title('피처 중요도 (전체 학습)', color='#13315C', fontweight='bold')
ax.set_facecolor('#F8FAF8')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('kinematic_importance.png', dpi=100, bbox_inches='tight')
plt.close()
print('저장: kinematic_importance.png')
print(importances)

저장: kinematic_importance.png
jerk_mean            0.152204
acceleration_mean    0.121704
jerk_std             0.107299
acceleration_std     0.101833
velocity_std         0.081515
velocity_cv          0.073399
total_distance       0.067067
velocity_mean        0.065378
jerk_max             0.064446
acceleration_max     0.063687
velocity_max         0.055158
duration             0.046310
dtype: float64


## 9. 모델 저장

In [60]:
MODEL_DIR = 'parkinson_web/models'
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, 'kinematic_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump({
        'pipeline':         pipeline,          # StandardScaler + RF
        'feature_cols':     FEATURE_COLS,
        'src_scalers':      final_src_scalers  # UCI/PaHaW 소스별 scaler (참고용)
    }, f)

print(f'저장 완료: {model_path}')
print(f'피처 ({len(FEATURE_COLS)}개): {FEATURE_COLS}')

저장 완료: parkinson_web/models\kinematic_model.pkl
피처 (12개): ['velocity_mean', 'velocity_std', 'velocity_max', 'velocity_cv', 'acceleration_mean', 'acceleration_std', 'acceleration_max', 'jerk_mean', 'jerk_std', 'jerk_max', 'total_distance', 'duration']
